# Nigeria COD-AB + COD-PS on admin-1 P-code
Uses only the Python standard library. Join humanitarian baseline layers on the official P-code, not the state name.

The sample includes `Lagos` vs `Lagos State` and `Federal Capital Territory` vs `FCT` so a name join fails while `adm1_pcode` matches.


In [ ]:
import csv, json
from pathlib import Path

root = Path(".")
table = json.loads((root / "data/nga-pcode-admin1.json").read_text())["byPcode"]
ab = {row["adm1_pcode"]: row for row in csv.DictReader((root / "data/samples/nga-cod-ab-sample.csv").open())}
ps = {row["adm1_pcode"]: row for row in csv.DictReader((root / "data/samples/nga-cod-ps-sample.csv").open())}

name_ab = {row["adm1_name"].lower(): row for row in ab.values()}
name_join = [p for p in ps.values() if p["adm1_name"].lower() in name_ab]
assert len(name_join) < len(ps), "name join should miss spelling variants"

joined = []
for pcode, boundary in ab.items():
    pop = ps.get(pcode)
    if not pop:
        continue
    assert pcode in table, f"unknown P-code {pcode}"
    joined.append((pcode, boundary["adm1_name"], int(pop["population"])))
assert len(joined) == len(ab) == len(ps)
keys = [row[0] for row in joined]
assert len(keys) == len(set(keys)), "P-code join is not one-to-one"
print("adm1_pcode adm1_name population")
for row in joined:
    print(*row)
print(f"{len(joined)} admin1 rows on P-code; name-only matched {len(name_join)}")
